# Step 3: Vessel Segmentation on Anchor Images

Method overview:
1. Build binary vessel masks for anchor images using a classical CV baseline.
2. Save outputs in evaluator-compatible format (.npz with key `mask`, plus .png).
3. Optionally evaluate using eval.py with grouping and registration skipped.

Outputs created:
- submission_step3/segmentation/anchor_XX.npz
- submission_step3/segmentation/anchor_XX.png
- submission_step3/segmentation_diagnostics.csv

In [9]:
from __future__ import annotations

import re
import subprocess
import sys
from pathlib import Path
from typing import Dict, List

import cv2
import numpy as np
import pandas as pd

from registration_utils import list_image_files, load_image, to_id

print('python:', sys.executable)
print('opencv:', cv2.__version__)

python: /Users/lucywu/quantum/miniconda3/envs/mia-final-1/bin/python
opencv: 4.13.0


## Configuration and Paths

In [32]:
anchors_dir_candidates = [
    Path('/Volumes/LUCY DISK/mia/Project 1/example test/anchor_images'),
    Path('/Users/lucywu/mia-final-1/data/example_test/anchor_images'),
    Path('/Users/lucywu/mia-final-1/anchor_images'),
]
gt_dir_candidates = [
    Path('/Volumes/LUCY DISK/mia/Project 1/example test/ground_truth'),
    Path('/Users/lucywu/mia-final-1/data/example_test/ground_truth'),
]
train_anchors_dir = Path('/Volumes/LUCY DISK/mia/Project 1/train/anchor_images')

submission_dir = Path('/Users/lucywu/mia-final-1/submission_step3')
segmentation_dir = submission_dir / 'segmentation'
diagnostics_csv = submission_dir / 'segmentation_diagnostics.csv'

# Scope modes: 'example_only', 'all_anchors', 'explicit_list'
scope_mode = 'example_only'
explicit_anchor_ids: List[str] = ['anchor_01', 'anchor_02']

# Baseline segmentation params (balanced precision/recall pass)
clahe_clip = 2.0
clahe_grid = (8, 8)
frangi_enabled = True
frangi_sigmas = (1, 2, 3, 4)
top_hat_kernel = 17
black_hat_kernel = 19
gaussian_blur = 3
adaptive_block_size = 31
adaptive_c = 3

# Threshold tuning knobs
q_high_percentile = 97
q_mid_percentile = 92
frangi_gate_percentile = 85
overfill_ratio = 0.20
q_tight_percentile = 99
fallback_percentile = 92

morph_open_kernel = 3
morph_close_kernel = 5
min_component_area = 12

def pick_readable_dir(candidates: List[Path], label: str, required: bool = True) -> Path | None:
    for p in candidates:
        if not p.exists():
            continue
        if not p.is_dir():
            continue
        try:
            _ = next(p.iterdir(), None)
            return p
        except PermissionError:
            print(f'Permission denied for {label}: {p}')
            continue

    if required:
        tried = '\n'.join([f'- {c}' for c in candidates])
        raise RuntimeError(
            f'Could not find a readable {label} directory.\n'
            f'Tried:\n{tried}\n\n'
            'Fix options:\n'
            '1) In macOS Settings -> Privacy & Security, allow VS Code/Python access to external volumes.\n'
            '2) Copy the dataset into a local workspace path and update candidates.'
        )
    return None

anchors_dir = pick_readable_dir(anchors_dir_candidates, label='anchors_dir', required=True)
gt_dir = pick_readable_dir(gt_dir_candidates, label='gt_dir', required=False)

submission_dir.mkdir(parents=True, exist_ok=True)
segmentation_dir.mkdir(parents=True, exist_ok=True)

print('anchors_dir:', anchors_dir)
print('gt_dir:', gt_dir)
print('submission_dir:', submission_dir)
print('segmentation_dir:', segmentation_dir)

anchors_dir: /Volumes/LUCY DISK/mia/Project 1/example test/anchor_images
gt_dir: /Volumes/LUCY DISK/mia/Project 1/example test/ground_truth
submission_dir: /Users/lucywu/mia-final-1/submission_step3
segmentation_dir: /Users/lucywu/mia-final-1/submission_step3/segmentation


## Segmentation Utilities

In [33]:
def get_example_anchor_ids_from_gt(gt_root: Path) -> List[str]:
    mask_dir = gt_root / 'mask'
    if not mask_dir.exists():
        return []
    ids: List[str] = []
    for p in sorted(mask_dir.glob('vessel_mask_*.tiff')):
        m = re.search(r'(\d+)', p.stem)
        if m:
            ids.append(f"anchor_{int(m.group(1)):02d}")
    return sorted(set(ids))


def estimate_fundus_mask(gray: np.ndarray) -> np.ndarray:
    blur = cv2.GaussianBlur(gray, (11, 11), 0)
    _, mask = cv2.threshold(blur, 10, 255, cv2.THRESH_BINARY)
    mask = cv2.medianBlur(mask, 9)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    return mask > 0


def frangi_vessel_enhance(gray: np.ndarray, sigmas: tuple[int, ...]) -> np.ndarray:
    g = gray.astype(np.float32) / 255.0
    responses: List[np.ndarray] = []
    for sigma in sigmas:
        ksize = max(3, 2 * int(3 * sigma) + 1)
        smoothed = cv2.GaussianBlur(g, (ksize, ksize), sigmaX=sigma, sigmaY=sigma)

        dxx = cv2.Sobel(smoothed, cv2.CV_32F, 2, 0, ksize=3)
        dyy = cv2.Sobel(smoothed, cv2.CV_32F, 0, 2, ksize=3)
        dxy = cv2.Sobel(smoothed, cv2.CV_32F, 1, 1, ksize=3)

        tmp = np.sqrt((dxx - dyy) ** 2 + 4.0 * (dxy ** 2))
        l1 = 0.5 * (dxx + dyy + tmp)
        l2 = 0.5 * (dxx + dyy - tmp)

        swap = np.abs(l1) < np.abs(l2)
        la = l1.copy()
        lb = l2.copy()
        la[swap], lb[swap] = l2[swap], l1[swap]

        rb = np.zeros_like(la)
        s2 = la ** 2 + lb ** 2
        valid = lb < 0

        beta = 0.5
        c = 0.1
        rb[valid] = np.exp(-(la[valid] ** 2) / (2.0 * (beta ** 2) * (lb[valid] ** 2 + 1e-12)))
        rb[valid] *= (1.0 - np.exp(-s2[valid] / (2.0 * (c ** 2))))
        responses.append(rb)

    vesselness = np.max(np.stack(responses, axis=0), axis=0)
    vesselness = cv2.normalize(vesselness, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return vesselness


def tophat_vessel_enhance(gray: np.ndarray, kernel_size: int = 17) -> np.ndarray:
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    top_hat = cv2.morphologyEx(gray, cv2.MORPH_TOPHAT, k)
    return cv2.normalize(top_hat, None, 0, 255, cv2.NORM_MINMAX)


def blackhat_vessel_enhance(gray: np.ndarray, kernel_size: int = 19) -> np.ndarray:
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (kernel_size, kernel_size))
    black_hat = cv2.morphologyEx(gray, cv2.MORPH_BLACKHAT, k)
    return cv2.normalize(black_hat, None, 0, 255, cv2.NORM_MINMAX)


def remove_small_components(mask01: np.ndarray, min_area: int) -> np.ndarray:
    n_lbl, labels, stats, _ = cv2.connectedComponentsWithStats(mask01.astype(np.uint8), connectivity=8)
    keep = np.zeros_like(mask01, dtype=np.uint8)
    for i in range(1, n_lbl):
        if stats[i, cv2.CC_STAT_AREA] >= min_area:
            keep[labels == i] = 1
    return keep


def segment_vessels_baseline(anchor_bgr: np.ndarray) -> np.ndarray:
    if anchor_bgr is None:
        raise ValueError('Input image is None')

    green = anchor_bgr[:, :, 1]
    clahe = cv2.createCLAHE(clipLimit=clahe_clip, tileGridSize=clahe_grid)
    enh = clahe.apply(green)

    if frangi_enabled:
        frangi_resp = frangi_vessel_enhance(enh, frangi_sigmas)
    else:
        frangi_resp = np.zeros_like(enh, dtype=np.uint8)

    tophat_resp = tophat_vessel_enhance(enh, top_hat_kernel)
    blackhat_resp = blackhat_vessel_enhance(enh, black_hat_kernel)

    vessel_resp = cv2.addWeighted(frangi_resp, 0.60, tophat_resp, 0.25, 0)
    vessel_resp = cv2.addWeighted(vessel_resp, 1.0, blackhat_resp, 0.15, 0)

    blur_k = max(1, gaussian_blur)
    if blur_k % 2 == 0:
        blur_k += 1
    vessel_resp = cv2.GaussianBlur(vessel_resp, (blur_k, blur_k), 0)

    resp_f = vessel_resp.astype(np.float32)
    q_high = np.percentile(resp_f, q_high_percentile)
    q_mid = np.percentile(resp_f, q_mid_percentile)
    frangi_q = max(np.percentile(frangi_resp, frangi_gate_percentile), 8)

    high_conf = resp_f >= q_high
    mid_conf = (resp_f >= q_mid) & (frangi_resp >= frangi_q)
    mask01 = (high_conf | mid_conf).astype(np.uint8)

    # Overfill guard: tighten threshold if too much of image is classified as vessel.
    if float(mask01.mean()) > overfill_ratio:
        q_tight = np.percentile(resp_f, q_tight_percentile)
        frangi_tight = max(np.percentile(frangi_resp, 95), 12)
        mask01 = ((resp_f >= q_tight) & (frangi_resp >= frangi_tight)).astype(np.uint8)

    # Sparse fallback only when almost empty.
    if float(mask01.mean()) < 0.001:
        cutoff = np.percentile(resp_f, fallback_percentile)
        mask01 = ((resp_f >= cutoff) & (frangi_resp >= frangi_q)).astype(np.uint8)

    fundus = estimate_fundus_mask(green)
    mask01 = (mask01 & fundus.astype(np.uint8)).astype(np.uint8)

    if morph_open_kernel > 1:
        ok = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_open_kernel, morph_open_kernel))
        mask01 = cv2.morphologyEx(mask01, cv2.MORPH_OPEN, ok)

    if morph_close_kernel > 1:
        ck = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (morph_close_kernel, morph_close_kernel))
        mask01 = cv2.morphologyEx(mask01, cv2.MORPH_CLOSE, ck)

    mask01 = remove_small_components(mask01, min_component_area)
    return (mask01 > 0).astype(np.uint8)


def select_anchor_paths(
    anchors_root: Path,
    mode: str,
    explicit_ids: List[str],
    gt_root: Path,
    ) -> List[Path]:
    paths = list_image_files(anchors_root)
    id_to_path: Dict[str, Path] = {to_id(p.name, 'anchor'): p for p in paths}

    if mode == 'all_anchors':
        selected_ids = sorted(id_to_path.keys())
    elif mode == 'explicit_list':
        selected_ids = sorted(set(explicit_ids))
    elif mode == 'example_only':
        selected_ids = get_example_anchor_ids_from_gt(gt_root)
        if not selected_ids:
            selected_ids = sorted(id_to_path.keys())
    else:
        raise ValueError(f'Unsupported scope_mode: {mode}')

    missing = [a for a in selected_ids if a not in id_to_path]
    if missing:
        raise FileNotFoundError(f'Anchor IDs not found in anchors_dir: {missing}')

    return [id_to_path[a] for a in selected_ids]


def save_segmentation_outputs(anchor_id: str, mask01: np.ndarray, out_dir: Path) -> Dict[str, str]:
    npz_path = out_dir / f'{anchor_id}.npz'
    png_path = out_dir / f'{anchor_id}.png'

    np.savez_compressed(npz_path, mask=mask01.astype(np.uint8))
    png_mask = (mask01.astype(np.uint8) * 255)
    ok = cv2.imwrite(str(png_path), png_mask)
    if not ok:
        raise IOError(f'Failed to save PNG: {png_path}')

    return {'npz': str(npz_path), 'png': str(png_path)}

## Run Segmentation Batch

In [30]:
anchor_paths = select_anchor_paths(
    anchors_root=anchors_dir,
    mode=scope_mode,
    explicit_ids=explicit_anchor_ids,
    gt_root=gt_dir,
)

print(f'Found {len(anchor_paths)} anchors to process (scope_mode={scope_mode}).')

rows: List[Dict[str, object]] = []

for p in anchor_paths:
    anchor_id = to_id(p.name, 'anchor')
    row: Dict[str, object] = {'anchor_id': anchor_id, 'image_path': str(p)}

    try:
        img = load_image(p)
        if img is None:
            raise ValueError('Unreadable image')

        mask01 = segment_vessels_baseline(img)
        outputs = save_segmentation_outputs(anchor_id, mask01, segmentation_dir)

        vessel_ratio = float(mask01.mean())
        row.update({
            'status': 'ok',
            'height': int(mask01.shape[0]),
            'width': int(mask01.shape[1]),
            'vessel_ratio': vessel_ratio,
            'npz_path': outputs['npz'],
            'png_path': outputs['png'],
            'error': '',
        })
    except Exception as exc:
        row.update({
            'status': 'error',
            'height': None,
            'width': None,
            'vessel_ratio': None,
            'npz_path': '',
            'png_path': '',
            'error': str(exc),
        })

    rows.append(row)

diag_df = pd.DataFrame(rows).sort_values('anchor_id')
diag_df.to_csv(diagnostics_csv, index=False)

ok_count = int((diag_df['status'] == 'ok').sum())
err_count = int((diag_df['status'] == 'error').sum())
print(f'Saved diagnostics: {diagnostics_csv}')
print(f'Completed: {ok_count} ok, {err_count} errors')
display(diag_df.head(10))

Found 5 anchors to process (scope_mode=example_only).
Saved diagnostics: /Users/lucywu/mia-final-1/submission_step3/segmentation_diagnostics.csv
Completed: 5 ok, 0 errors


,anchor_id,image_path,status,height,width,vessel_ratio,npz_path,png_path,error
0,anchor_01,/Volumes/LUCY DISK/mia/Project 1/example test/...,ok,2912,2912,0.033711,/Users/lucywu/mia-final-1/submission_step3/seg...,/Users/lucywu/mia-final-1/submission_step3/seg...,
1,anchor_02,/Volumes/LUCY DISK/mia/Project 1/example test/...,ok,2912,2912,0.026122,/Users/lucywu/mia-final-1/submission_step3/seg...,/Users/lucywu/mia-final-1/submission_step3/seg...,
2,anchor_03,/Volumes/LUCY DISK/mia/Project 1/example test/...,ok,2912,2912,0.034799,/Users/lucywu/mia-final-1/submission_step3/seg...,/Users/lucywu/mia-final-1/submission_step3/seg...,
3,anchor_04,/Volumes/LUCY DISK/mia/Project 1/example test/...,ok,2912,2912,0.023819,/Users/lucywu/mia-final-1/submission_step3/seg...,/Users/lucywu/mia-final-1/submission_step3/seg...,
4,anchor_05,/Volumes/LUCY DISK/mia/Project 1/example test/...,ok,2912,2912,0.035409,/Users/lucywu/mia-final-1/submission_step3/seg...,/Users/lucywu/mia-final-1/submission_step3/seg...,


## Optional Evaluation (Segmentation Only)

In [31]:
if gt_dir is not None and gt_dir.exists():
    try:
        from PIL import Image  # noqa: F401
    except ModuleNotFoundError:
        print('Pillow is missing in this environment. Installing now...')
        subprocess.run([sys.executable, '-m', 'pip', 'install', 'Pillow'], check=True)

    subprocess.run([
        sys.executable,
        '/Users/lucywu/mia-final-1/eval.py',
        '--pred_dir', str(submission_dir),
        '--gt_dir', str(gt_dir),
        '--skip', 'grouping', 'registration',
        '-v',
    ], check=True)
else:
    print('Skipping eval because gt_dir is not configured/readable.')

  MIA 2026 Project 1 — Evaluation Results

  Task 1: Anchor Assignment (Grouping)
------------------------------------------------------------------------
  [SKIPPED]

  Task 2: Warped Coordinate Prediction (Registration)
------------------------------------------------------------------------
  [SKIPPED]

  Task 3: Vessel Segmentation
------------------------------------------------------------------------
  TPR:       0.2528  (78156/309152 vessel pixels)
  TNR:       0.9758  (1465025/1501390 background pixels)
  Evaluated: 5/5 anchor images

  Per-anchor results:
    anchor_01:  TPR = 0.3555  (10786/30342),  TNR = 0.9685  (161669/166921)
    anchor_02:  TPR = 0.0455  (1429/31420),  TNR = 0.9633  (250873/260420)
    anchor_03:  TPR = 0.1736  (18106/104281),  TNR = 0.9625  (192040/199517)
    anchor_04:  TPR = 0.3158  (23753/75207),  TNR = 0.9895  (393998/398189)
    anchor_05:  TPR = 0.3547  (24082/67902),  TNR = 0.9792  (466445/476343)

  Summary
-------------------------------------